# 01 — Exploratory Data Analysis

Goal: understand the dataset before we engineer features or train models.

- Class balance (benign vs tunneling)
- Distribution of raw query lengths
- Initial entropy distribution
- Sample queries from each class

**Data**: this notebook works on any CSV with a query column (`query`/`domain`/`name`) and a binary label column.  
If you have not downloaded the CIC datasets yet, run `python -m src.generate_sample` from `dns-tunneling/` to create a synthetic stand-in at `data/raw/sample.csv`.

In [ ]:
import sys
from pathlib import Path

# Make the `src` package importable when running from notebooks/.
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.preprocess import load_dataset
from src.features import shannon_entropy, query_length

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
# Point this at any CSV / dir of CSVs you have.
DATA_PATH = ROOT / "data" / "raw" / "sample.csv"
data, labels = load_dataset(DATA_PATH)
print(f"rows: {len(data):,}")
data.head()

## Class balance

In [ ]:
counts = labels.value_counts().rename({0: "benign", 1: "tunneling"})
print(counts)
ax = counts.plot(kind="bar", color=["#4c72b0", "#dd8452"])
ax.set_title("Class distribution")
ax.set_ylabel("count")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.show()

## Query-length distribution

Tunneling queries pack payload data into subdomains, so they tend to be **much longer** than benign ones.

In [ ]:
if "query" in data.columns:
    df = data.copy()
    df["length"] = df["query"].map(query_length)
    df["label"] = labels.values
    sns.histplot(
        data=df, x="length", hue="label", bins=50, kde=True, palette="Set1"
    )
    plt.title("Query length by class")
    plt.show()
else:
    print("Dataset is already pre-featurised — no raw `query` column.")

## Entropy distribution

Base32/base64-encoded payloads have higher per-character entropy than typical English-like domain names.

In [ ]:
if "query" in data.columns:
    df["entropy"] = df["query"].map(shannon_entropy)
    sns.histplot(
        data=df, x="entropy", hue="label", bins=50, kde=True, palette="Set1"
    )
    plt.title("Shannon entropy by class")
    plt.show()

## Side-by-side examples

In [ ]:
if "query" in data.columns:
    print("Benign samples:")
    for q in data.loc[labels == 0, "query"].sample(5, random_state=0):
        print("  ", q)
    print("\nTunneling samples:")
    for q in data.loc[labels == 1, "query"].sample(5, random_state=0):
        print("  ", q)

## Takeaways

1. The two classes separate cleanly on length and entropy — strong signal for a classifier.
2. The dataset is roughly balanced (or close), so we can train without resampling.
3. Next step: extract the full feature set and look at correlations → `02_feature_engineering.ipynb`.